# Pertemuan 5 — Praktikum 2: Penerapan K-Means

Praktikum ini membahas data buatan, pengelompokan angka, dan pengurangan warna gambar. Dataset angka dan gambar bunga sudah tersedia di scikit-learn, sehingga tidak perlu diunduh terpisah.

Sumber: [Praktikum 2](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js04-klasterisasi/praktikum-2). Jalankan sel dari atas ke bawah.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, SpectralClustering, MiniBatchKMeans
from sklearn.datasets import make_blobs, make_moons, load_digits, load_sample_image
from sklearn.metrics import pairwise_distances_argmin, accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
from scipy.stats import mode

sns.set_theme(style='whitegrid')


## 1. K-Means pada data buatan
Kita membuat 300 titik dengan empat kelompok. Model hanya menggunakan koordinat `X`, bukan label asli.


In [2]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=50)
plt.title('Data Buatan Sebelum Klasterisasi')
plt.show()

kmeans = KMeans(n_clusters=4, random_state=0, n_init=10)
y_kmeans = kmeans.fit_predict(X)
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, s=50, cmap='viridis')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='black', marker='X', s=200, label='Centroid')
plt.title('Hasil K-Means dengan Empat Klaster')
plt.legend()
plt.show()


## 2. Menghitung klaster secara bertahap
Prosesnya adalah memilih centroid awal, memasukkan titik ke centroid terdekat, lalu menghitung ulang centroid. Langkah ini diulang sampai posisi centroid stabil. Modul menyebutnya Expectation-Maximization; kode berikut menjalankan pembaruan K-Means sederhana.

Batas iterasi mencegah pengulangan terlalu lama. Jika ada klaster kosong, centroid sebelumnya tetap dipakai.


In [3]:
def find_clusters(X, n_clusters, rseed=2):
    rng = np.random.RandomState(rseed)
    centers = X[rng.permutation(len(X))[:n_clusters]].copy()
    for _ in range(300):
        labels = pairwise_distances_argmin(X, centers)
        new_centers = centers.copy()
        for i in range(n_clusters):
            members = X[labels == i]
            if len(members) > 0:
                new_centers[i] = members.mean(axis=0)
        if np.allclose(centers, new_centers):
            centers = new_centers
            break
        centers = new_centers
    return centers, pairwise_distances_argmin(X, centers)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, seed in zip(axes, [2, 0]):
    centers, labels = find_clusters(X, 4, rseed=seed)
    ax.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis')
    ax.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=150)
    ax.set_title(f'Centroid awal dengan random seed = {seed}')
plt.tight_layout()
plt.show()


Posisi centroid awal dapat menghasilkan pembagian yang berbeda. Karena itu, K-Means biasanya dicoba dari beberapa posisi awal.


## 3. Mengubah jumlah klaster
Saat k dinaikkan menjadi enam, empat kelompok awal dapat terpecah. Jumlah klaster yang lebih banyak belum tentu lebih sesuai dengan struktur data.


In [4]:
labels6 = KMeans(n_clusters=6, random_state=0, n_init=10).fit_predict(X)
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels6, s=50, cmap='viridis')
plt.title('Data yang Sama dengan Enam Klaster')
plt.show()


## 4. Data dengan bentuk melengkung
`make_moons` membentuk dua kelompok seperti bulan sabit. Kita membandingkan K-Means dan Spectral Clustering. K-Means memakai jarak terhadap centroid, sedangkan Spectral Clustering melihat hubungan antar titik yang berdekatan.

Jika muncul peringatan bahwa graf tidak terhubung penuh, itu berkaitan dengan kelompok tetangga yang terpisah pada data ini.


In [5]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=0)
labels_km = KMeans(n_clusters=2, random_state=0, n_init=10).fit_predict(X_moons)
spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors',
                             assign_labels='kmeans', random_state=0)
labels_sp = spectral.fit_predict(X_moons)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, labels, title in zip(axes, [y_moons, labels_km, labels_sp],
                              ['Kelompok Asli', 'K-Means', 'Spectral Clustering']):
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, s=35, cmap='viridis')
    ax.set_title(title)
plt.tight_layout()
plt.show()


/private/tmp/ml-pertemuan5-venv/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


## 5. Mengelompokkan gambar angka
Dataset digits berisi 1.797 gambar angka ukuran 8 × 8 piksel. Setiap gambar menjadi 64 fitur. Kita membentuk sepuluh klaster, lalu menampilkan centroid sebagai gambar.


In [6]:
digits = load_digits()
print('Ukuran data:', digits.data.shape)
kmeans_digits = KMeans(n_clusters=10, random_state=0, n_init=10)
clusters = kmeans_digits.fit_predict(digits.data)
print('Ukuran centroid:', kmeans_digits.cluster_centers_.shape)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, (ax, center) in enumerate(zip(axes.flat, kmeans_digits.cluster_centers_.reshape(10, 8, 8))):
    ax.imshow(center, cmap='binary', interpolation='nearest')
    ax.set_title(f'Klaster {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()


Ukuran data: (1797, 64)
Ukuran centroid: (10, 64)


### Evaluasi dengan label angka
Nomor klaster belum menunjukkan angka tertentu. Untuk evaluasi, setiap klaster diberi label angka yang paling sering muncul di dalamnya. Label asli hanya dipakai setelah klasterisasi.

Akurasi berikut dihitung pada data yang sama dan pemetaan label memakai data tersebut. Jadi, nilainya merupakan evaluasi pengelompokan, bukan akurasi model pada data uji baru.


In [7]:
def label_mayoritas(clusters, target):
    labels = np.zeros_like(clusters)
    for cluster in np.unique(clusters):
        mask = clusters == cluster
        labels[mask] = mode(target[mask], keepdims=False).mode
    return labels

labels_digits = label_mayoritas(clusters, digits.target)
akurasi_awal = accuracy_score(digits.target, labels_digits)
print(f'Akurasi pemetaan klaster: {akurasi_awal:.2%}')
mat = confusion_matrix(digits.target, labels_digits)
plt.figure(figsize=(8, 7))
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=digits.target_names, yticklabels=digits.target_names)
plt.xlabel('Label hasil pemetaan')
plt.ylabel('Label asli')
plt.title('Confusion Matrix K-Means pada Digits')
plt.show()


Akurasi pemetaan klaster: 79.19%


### Menggunakan t-SNE
t-SNE mengubah 64 fitur menjadi dua dimensi. Setelah itu, kita menjalankan K-Means lagi. Proses ini dapat memerlukan waktu lebih lama daripada sel lainnya.


In [8]:
tsne = TSNE(n_components=2, init='random', random_state=0, learning_rate='auto')
digits_proj = tsne.fit_transform(digits.data)
model_tsne = KMeans(n_clusters=10, random_state=0, n_init=10)
clusters_tsne = model_tsne.fit_predict(digits_proj)
labels_tsne = label_mayoritas(clusters_tsne, digits.target)
akurasi_tsne = accuracy_score(digits.target, labels_tsne)
display(pd.DataFrame({'Metode': ['K-Means pada 64 fitur', 't-SNE lalu K-Means'],
                      'Akurasi pemetaan': [akurasi_awal, akurasi_tsne]}))
plt.figure(figsize=(8, 5))
plt.scatter(digits_proj[:, 0], digits_proj[:, 1], c=clusters_tsne, cmap='tab10', s=10)
plt.title('Klaster Angka Setelah t-SNE')
plt.xlabel('Dimensi t-SNE 1')
plt.ylabel('Dimensi t-SNE 2')
plt.show()


## 6. Mengurangi warna gambar menjadi 16 warna
Setiap piksel mempunyai tiga nilai: merah, hijau, dan biru. Kita mengelompokkan warna piksel menjadi 16 kelompok dengan MiniBatchKMeans. Warna setiap piksel kemudian diganti dengan warna centroid kelompoknya.

Ini mengurangi jumlah warna, tetapi belum otomatis mengecilkan ukuran file gambar. Ukuran file juga bergantung pada format penyimpanan.


In [9]:
flower = load_sample_image('flower.jpg')
plt.figure(figsize=(8, 5))
plt.imshow(flower)
plt.title('Gambar Bunga Asli')
plt.axis('off')
plt.show()
print('Ukuran gambar:', flower.shape)
data = (flower / 255.0).reshape(-1, 3)
print('Ukuran data piksel:', data.shape)


Ukuran gambar: (427, 640, 3)
Ukuran data piksel: (273280, 3)


In [10]:
def plot_pixels(data, title, colors=None, N=10000):
    if colors is None:
        colors = data
    rng = np.random.RandomState(0)
    indices = rng.permutation(len(data))[:N]
    R, G, B = data[indices].T
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(R, G, c=colors[indices], marker='.', s=8)
    axes[1].scatter(R, B, c=colors[indices], marker='.', s=8)
    axes[0].set(xlabel='Merah', ylabel='Hijau', xlim=(0, 1), ylim=(0, 1))
    axes[1].set(xlabel='Merah', ylabel='Biru', xlim=(0, 1), ylim=(0, 1))
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_pixels(data, 'Sebaran Warna Asli')
model_warna = MiniBatchKMeans(n_clusters=16, random_state=0, n_init=10)
model_warna.fit(data)
new_colors = model_warna.cluster_centers_[model_warna.predict(data)]
plot_pixels(data, 'Sebaran Setelah Dikurangi Menjadi 16 Warna', colors=new_colors)
flower_recolored = new_colors.reshape(flower.shape)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, gambar, judul in zip(axes, [flower, flower_recolored], ['Gambar Asli', 'Gambar 16 Warna']):
    ax.imshow(gambar)
    ax.set_title(judul)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Jumlah warna hasil:', len(np.unique(new_colors, axis=0)))


Jumlah warna hasil: 16


## Kesimpulan
K-Means dapat digunakan untuk mengelompokkan titik, gambar angka, dan warna piksel. Hasilnya dipengaruhi jumlah klaster, posisi awal centroid, dan bentuk data. Pada data bulan sabit, kita bisa melihat keterbatasan K-Means melalui perbandingan dengan Spectral Clustering. Pengurangan warna membuat gambar tetap dikenali dengan lebih sedikit warna.

Pada pengecekan ini, akurasi pemetaan klaster angka sekitar 79,19% sebelum t-SNE dan 94,10% setelah t-SNE. Gambar bunga berhasil dikurangi menjadi 16 warna. Nilai akurasi tersebut memakai data yang sama untuk pemetaan dan evaluasi.
